# Exp 2 — Where exactly does padding leak in, and can we remove it without retraining?

Exp 1 established that 30–46% of images change their detections with their batch-mates, that the content of the batch-mates is irrelevant, and that the effect switches on with even 5% padding. This notebook does two things.

**Part A — Fine padding sweep.** 200 images, one at a time, with 0 → 256 pixels of padding added on the right and bottom. Where exactly does the switch flip? The sweep is run twice: without and with the fix.

**Part B — The fix, component by component.** Four detectors (DETR, Faster R-CNN, RetinaNet, FCOS) on the same 500 COCO images. Each fix component (Section 3) is switched on alone, then all together (`full`), at batch size 1 and with random batches of 8. We measure:
- **invariance:** does batch 8 give the same answer as batch 1 *under the same fix*? (target ≈ 0% flips)
- **fidelity:** AP and flips against the library's normal batch-1 run;
- **speed:** does the fix keep batching fast?

**Claims (fixed before running):**
- **E1 (step):** without the fix, the smallest padding step already produces ≥ 50% of the flip rate seen at 256 px.
- **E2 (invariance):** with `full`, flips between batch 1 and batch 8 are ≤ 0.5% for every model, in both the random and the max-padding batches.
- **E3 (fidelity):** with `full`, batch-8 AP is within 0.3 of the normal batch-1 AP for every model.
- **E4 (speed):** `full` at batch 8 keeps ≥ 90% of the normal batch-8 speed.

**Kaggle:** GPU T4 x2 or P100, Internet ON, nothing to attach. About 1.5–2 hours. Results are saved per model, so a restart resumes where it stopped.

In [ ]:
!pip install -q "transformers==5.17.0" pycocotools timm

In [ ]:
import os
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")   # less memory fragmentation
import io, sys, glob, json, time, math, random, zipfile, platform, datetime, contextlib, urllib.request, dataclasses, gc
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, torch, torchvision
import torch.nn.functional as F
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
import transformers
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

OUT = "/kaggle/working/exp2"
os.makedirs(OUT, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

ENV = {
    "run_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(json.dumps(ENV, indent=2))
if ENV["gpu"] is None:
    print("\n⚠⚠ NO GPU. Stop, set Accelerator to GPU in the right panel, and Run All again. ⚠⚠")

def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()

## 1. Configuration

In [ ]:
CFG = {
    "n_images": 500,
    "seed": 0,
    "batch_size": 8,
    # Part A: fine padding sweep (single image, padding added on right+bottom, in model-input pixels)
    "sweep_models": ["detr_r50", "frcnn_v2", "retinanet_v2"],
    "n_images_sweep": 200,
    "sweep_px": {"hf": [0, 1, 2, 4, 8, 16, 32, 64, 128, 256], "tv": [0, 32, 64, 128, 256]},
    # Part B: fix evaluation on all 500 images
    "fix_models": ["detr_r50", "frcnn_v2", "retinanet_v2", "fcos"],
    "fix_margin": 32,
    "fix_rpn_thresh": 1e-6,
    "micro_batch": {"deformable_detr": 2},
    "flip_score_thr": 0.5,
    "flip_iou": 0.9,
    "coco_root": "/kaggle/input",
    "coco_download_dir": "/kaggle/working/coco",
    "claims": {"E2_flip": 0.005, "E3_dap": 0.3, "E4_speed": 0.90},
}
RUN_PARTS = {"A": True, "B": True}
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])

## 2. COCO val2017 (same 500 images as Day 0 and Exp 1)

In [ ]:
def find_attached_coco(root):
    anns = glob.glob(os.path.join(root, "**", "instances_val2017.json"), recursive=True)
    probe = glob.glob(os.path.join(root, "**", "000000000139.jpg"), recursive=True)
    if anns and probe:
        return anns[0], os.path.dirname(probe[0])
    return None, None

ANN, IMG_DIR = find_attached_coco(CFG["coco_root"])
if ANN is None:
    os.makedirs(CFG["coco_download_dir"], exist_ok=True)
    ANN = os.path.join(CFG["coco_download_dir"], "instances_val2017.json")
    IMG_DIR = os.path.join(CFG["coco_download_dir"], "val2017")
    if not os.path.exists(ANN):
        zp = os.path.join(CFG["coco_download_dir"], "ann.zip")
        print("downloading COCO annotations (~250 MB) ...")
        urllib.request.urlretrieve("http://images.cocodataset.org/annotations/annotations_trainval2017.zip", zp)
        with zipfile.ZipFile(zp) as z, open(ANN, "wb") as f:
            f.write(z.read("annotations/instances_val2017.json"))
        os.remove(zp)

with contextlib.redirect_stdout(io.StringIO()):
    GT = COCO(ANN)
NAME2CAT = {c["name"].strip().lower(): c["id"] for c in GT.loadCats(GT.getCatIds())}
IDS = sorted(random.Random(CFG["seed"]).sample(sorted(GT.getImgIds()), CFG["n_images"]))
INFO = GT.loadImgs(IDS)
os.makedirs(IMG_DIR, exist_ok=True)
def fetch(info):
    p = os.path.join(IMG_DIR, info["file_name"])
    if not os.path.exists(p):
        urllib.request.urlretrieve(f"http://images.cocodataset.org/val2017/{info['file_name']}", p)
    return p
with ThreadPoolExecutor(16) as ex:
    paths = list(ex.map(fetch, INFO))
IMGS = [Image.open(p).convert("RGB") for p in paths]
N = len(IMGS)
print(f"{N} images; first ids {IDS[:3]} (Day 0 used seed 0 too, so these are the same images)")

## 3. The fix, as switchable components

Reading the model code gave four places where the padded canvas leaks into an image's prediction. Each gets its own switch, so we can measure how much churn each one causes. **`full`** = all switches that apply to a model.

| switch | leak it removes | models |
|---|---|---|
| `conv` | Padding pixels become non-zero features after the first layer, so the image border stops looking like a border. We re-zero the padding at the input of every conv / pool, zero it in the backbone outputs, and drop head scores at padded positions. | all |
| `norm` | GroupNorm averages over the whole feature map, padding included, so *every* feature changes. We compute statistics over the valid region only. | RetinaNet, FCOS |
| `strides` | torchvision places anchors at `canvas // grid` (integer division), so anchor spacing changes with the canvas. We use the nominal power-of-two stride. | torchvision models |
| `exactmask` | DETR shrinks its padding mask with nearest-neighbour rounding, which can misplace the valid/padded boundary by one row. We compute the exact extents. | DETR |

**Two small side effects of `conv`, applied identically at batch 1 and batch 8:** torchvision models get a fixed 32-pixel margin, so every image has at least one empty row and column past its edge; and RPN proposals with a probability below 1e-6 are dropped. Both keep the computation the same regardless of batch-mates.

All four switches are training-free and need no new weights. `full` can change batch-1 outputs a little for torchvision models, because their normal batch-1 run already pads to a multiple of 32. For DETR, `full` at batch 1 is exactly the normal batch 1.

In [ ]:
class Fix:
    """Switchable components that make a detector's output independent of the padded canvas.
    All switched off = the library's normal behaviour. `valid` (per-image unpadded size in model-input pixels)
    and `canvas` (padded size) are set by the detector right before each forward pass."""
    COMPONENTS = ("conv", "norm", "strides", "exactmask")
    def __init__(self):
        for c in self.COMPONENTS:
            setattr(self, c, False)
        self.valid, self.canvas, self._cache = None, None, {}

    def set(self, components):
        for c in self.COMPONENTS:
            setattr(self, c, c in components)

    def applies(self, x):
        return self.valid is not None and x.dim() == 4 and x.shape[0] == len(self.valid)

    @staticmethod
    def _stride(n, N):
        # every stride-s layer in these backbones maps size N -> ceil(N/s)
        for s in (1, 2, 4, 8, 16, 32, 64, 128, 256):
            if math.ceil(N / s) == n:
                return s
        return N / n

    def mask(self, x):
        """(B,1,H,W) tensor: 1 inside each image's valid region at x's resolution, 0 in the padding."""
        H, W = x.shape[-2:]
        key = (H, W, tuple(self.valid), tuple(self.canvas), x.device, x.dtype)
        if key not in self._cache:
            sh, sw = self._stride(H, self.canvas[0]), self._stride(W, self.canvas[1])
            m = torch.zeros(len(self.valid), 1, H, W, device=x.device, dtype=x.dtype)
            for i, (h, w) in enumerate(self.valid):
                m[i, :, :math.ceil(h / sh), :math.ceil(w / sw)] = 1
            if len(self._cache) > 64:
                self._cache.clear()
            self._cache[key] = m
        return self._cache[key]

def _mask_conv_inputs(root, fix):
    """conv: zero the padding at the input of every conv / max-pool, so each layer sees exactly the zeros an
    unpadded image would see at its border (after ReLU all activations are >= 0, so this also holds for max-pool)."""
    def pre(mod, inp):
        x = inp[0]
        if fix.conv and fix.applies(x):
            return (x * fix.mask(x),) + tuple(inp[1:])
    n = 0
    for m in root.modules():
        if isinstance(m, (nn.Conv2d, nn.MaxPool2d)):
            m.register_forward_pre_hook(pre); n += 1
    return n

def _masked_groupnorm(root, fix):
    """norm: GroupNorm statistics over the valid region only (normal GroupNorm averages over the padding too)."""
    n = 0
    for mod in root.modules():
        if isinstance(mod, nn.GroupNorm):
            orig = mod.forward
            def fwd(x, mod=mod, orig=orig):
                if not (fix.norm and fix.applies(x)):
                    return orig(x)
                m = fix.mask(x); B, C, H, W = x.shape; G = mod.num_groups
                xg, mg = x.reshape(B, G, C // G, H, W), m.reshape(B, 1, 1, H, W)
                cnt = mg.sum(dim=(2, 3, 4), keepdim=True) * (C // G)
                mean = (xg * mg).sum(dim=(2, 3, 4), keepdim=True) / cnt
                var = (((xg - mean) ** 2) * mg).sum(dim=(2, 3, 4), keepdim=True) / cnt
                y = ((xg - mean) / torch.sqrt(var + mod.eps)).reshape(B, C, H, W)
                if mod.affine:
                    y = y * mod.weight[None, :, None, None] + mod.bias[None, :, None, None]
                return y
            mod.forward = fwd; n += 1
    return n

def _nominal_strides(ag, fix):
    """strides: torchvision sets anchor spacing to canvas_size // grid_size (integer division), which changes
    with the canvas. Use the nominal power-of-two stride instead."""
    orig = ag.forward
    def fwd(image_list, feature_maps):
        if not fix.strides:
            return orig(image_list, feature_maps)
        grid_sizes = [f.shape[-2:] for f in feature_maps]
        H, W = image_list.tensors.shape[-2:]
        dtype, device = feature_maps[0].dtype, feature_maps[0].device
        p2 = lambda r: torch.tensor(int(2 ** round(math.log2(r))), dtype=torch.int64, device=device)
        strides = [[p2(H / g[0]), p2(W / g[1])] for g in grid_sizes]
        ag.set_cell_anchors(dtype, device)
        a = ag.grid_anchors(grid_sizes, strides)
        return [torch.cat(a) for _ in image_list.image_sizes]
    ag.forward = fwd

def _suppress_padding_scores(conv, fix):
    """conv: classification logits at padded positions -> -1e4 (anchors there would not exist without padding)."""
    def hook(mod, inp, out):
        if fix.conv and fix.applies(out):
            return out.masked_fill(fix.mask(out) == 0, -1e4)
    conv.register_forward_hook(hook)

def install_fix_tv(model, fix):
    n = _mask_conv_inputs(model.backbone, fix)
    heads = [model.rpn.head] if hasattr(model, "rpn") else [model.head]
    for h in heads:
        n += _mask_conv_inputs(h, fix)
    def zero_outputs(mod, inp, out):
        if fix.conv and fix.valid is not None:
            return type(out)((k, v * fix.mask(v)) for k, v in out.items())
    model.backbone.register_forward_hook(zero_outputs)
    g = sum(_masked_groupnorm(h, fix) for h in heads) + _masked_groupnorm(model.backbone, fix)
    if hasattr(model, "rpn"):
        _nominal_strides(model.rpn.anchor_generator, fix); _suppress_padding_scores(model.rpn.head.cls_logits, fix)
    else:
        _nominal_strides(model.anchor_generator, fix); _suppress_padding_scores(model.head.classification_head.cls_logits, fix)
    fix.available = ("conv", "norm", "strides") if g else ("conv", "strides")

def install_fix_hf(model, fix):
    enc = model.model.backbone
    n = _mask_conv_inputs(enc, fix)
    orig = enc.forward
    def fwd(pixel_values, pixel_mask):
        out = orig(pixel_values, pixel_mask)
        if not (fix.exactmask and fix.valid is not None):
            return out
        # exactmask: the transformer's padding mask with exact ceil(h/s) extents (the library uses nearest resize)
        return [(fm, fix.mask(fm)[:, 0].bool()) for fm, _ in out]
    enc.forward = fwd
    fix.available = ("conv", "exactmask")

## 4. Detectors (same wrappers as Exp 1, with the fix switches wired in)

In [ ]:
ALIASES = {"motorbike": "motorcycle", "aeroplane": "airplane", "sofa": "couch", "tvmonitor": "tv", "tv monitor": "tv",
           "diningtable": "dining table", "pottedplant": "potted plant", "hair dryer": "hair drier",
           "sports ball": "sports ball", "cellphone": "cell phone", "mobile phone": "cell phone"}
def _norm(s):
    s = " ".join(str(s).lower().replace("_", " ").replace("-", " ").split())
    return ALIASES.get(s, s)
def names_to_cat(names, label=""):
    m, miss = {}, []
    for k, v in names.items():
        c = NAME2CAT.get(_norm(v))
        if c is not None:
            m[int(k)] = c
        elif _norm(v) not in ("n/a", "__background__", "background", "no object"):
            miss.append(v)
    if miss:
        print(f"  unmapped class names in {label}: {miss}")
    return m

def load_hf_model(cls, repo, revision=None):
    try:
        return cls.from_pretrained(repo, revision=revision)
    except Exception as ex:
        if "alidation" not in f"{type(ex).__name__} {ex}":
            raise
        from huggingface_hub import hf_hub_download
        raw = json.load(open(hf_hub_download(repo, "config.json", revision=revision)))
        defaults = {f.name: f.default for f in dataclasses.fields(cls.config_class) if f.default is not dataclasses.MISSING}
        dropped = [k for k, v in raw.items() if v is None and defaults.get(k) is not None]
        print(f"  (config fix for {repo}: replaced null fields {dropped} with library defaults)")
        raw = {k: v for k, v in raw.items() if k not in dropped}
        return cls.from_pretrained(repo, revision=revision, config=cls.config_class.from_dict(raw))

def roundup(x, d):
    return int(math.ceil(x / d) * d)

def read_csv(path):
    try:
        return pd.read_csv(path)
    except (FileNotFoundError, pd.errors.EmptyDataError):
        return pd.DataFrame()

def _pack(boxes, scores, labels, lab2cat):
    lab = labels.cpu().numpy()
    keep = np.array([l in lab2cat for l in lab], bool)
    return {"boxes": boxes.float().cpu().numpy()[keep], "scores": scores.float().cpu().numpy()[keep],
            "labels": np.array([lab2cat[l] for l in lab[keep]], dtype=int)}

class MicroBatch:
    """If a batch does not fit in GPU memory, split it into smaller pieces that are all padded to the canvas the
    full batch would have had. Images in these models never interact except through padding, so this gives the
    same result as the full batch (verified for DETR by the canvas self-test, and for every Part-B model by B0)."""
    micro = None   # largest piece size known to fit; None = no limit found yet

    def batch_canvas(self, imgs):
        sizes = [self.input_size(im) for im in imgs]
        return (roundup(max(h for h, _ in sizes), self.div), roundup(max(w for _, w in sizes), self.div))

    def run(self, imgs, canvas=None):
        if self.fixed_size:
            return self._run(imgs, canvas)
        if self.micro is not None and len(imgs) > self.micro:
            return self._split(imgs, canvas)
        try:
            return self._run(imgs, canvas)
        except torch.cuda.OutOfMemoryError:
            if len(imgs) == 1:
                raise
        # We are now OUTSIDE the except block, so the failed attempt (and the GPU memory it holds) is released
        # before retrying. Retrying inside the except block keeps that memory alive.
        free_mem()
        self.micro = max(1, len(imgs) // 2)
        print(f"  [{self.name}] batch of {len(imgs)} does not fit in GPU memory → running pieces of {self.micro} "
              f"on the full batch's canvas (equivalent result)")
        return self._split(imgs, canvas)

    def _split(self, imgs, canvas):
        canvas = canvas if canvas is not None else self.batch_canvas(imgs)
        step = self.micro                      # fixed for this call, even if a piece lowers self.micro further
        R, P = [], []
        for k in range(0, len(imgs), step):
            r, p = self.run(imgs[k:k + step], canvas)
            R += r; P += p
        assert len(R) == len(imgs)
        return R, P

class HFDet(MicroBatch):
    kind, div = "hf", 1
    def __init__(self, name, repo, cls_name, revision=None, fixed_size=False):
        from transformers import AutoImageProcessor
        self.name, self.fixed_size = name, fixed_size
        self.proc = AutoImageProcessor.from_pretrained(repo, revision=revision)
        self.model = load_hf_model(getattr(transformers, cls_name), repo, revision).to(DEVICE).eval()
        self.lab2cat = names_to_cat(self.model.config.id2label, name)
        self.fix = Fix()
        if not fixed_size:
            install_fix_hf(self.model, self.fix)

    def _single(self, img):
        enc = self.proc(images=[img], return_tensors="pt")
        pv = enc["pixel_values"][0]
        m = enc["pixel_mask"][0] if "pixel_mask" in enc else torch.ones(pv.shape[-2:], dtype=torch.long)
        return pv, m

    def input_size(self, img):
        return tuple(self._single(img)[0].shape[-2:])

    @torch.inference_mode()
    def _run(self, imgs, canvas=None):
        if canvas is None:
            enc = self.proc(images=imgs, return_tensors="pt")
            pv = enc["pixel_values"]
            mask = enc["pixel_mask"] if "pixel_mask" in enc else None
        else:
            H, W = canvas; pvs, ms = [], []
            for im in imgs:
                p, m = self._single(im); h, w = p.shape[-2:]
                assert h <= H and w <= W, f"canvas {canvas} smaller than image {(h, w)}"
                pvs.append(F.pad(p, (0, W - w, 0, H - h))); ms.append(F.pad(m, (0, W - w, 0, H - h)))
            pv, mask = torch.stack(pvs), torch.stack(ms)
        kw = {"pixel_values": pv.to(DEVICE)}
        if mask is not None:
            kw["pixel_mask"] = mask.to(DEVICE)
            self.fix.valid = [(int(mask[i, :, 0].sum()), int(mask[i, 0, :].sum())) for i in range(len(imgs))]
            self.fix.canvas = tuple(pv.shape[-2:])
        try:
            out = self.model(**kw)
        finally:
            self.fix.valid = None
        sizes = torch.tensor([[im.size[1], im.size[0]] for im in imgs])
        res = self.proc.post_process_object_detection(out, threshold=0.0, target_sizes=sizes)
        pad = (1 - mask.float().mean(dim=(1, 2))).tolist() if mask is not None else [0.0] * len(imgs)
        return [_pack(r["boxes"], r["scores"], r["labels"], self.lab2cat) for r in res], pad

class TVDet(MicroBatch):
    kind, div, fixed_size = "tv", 32, False
    def __init__(self, name, builder, weights):
        self.name = name
        self.model = builder(weights=weights).to(DEVICE).eval()
        self.tf = weights.transforms()
        self.lab2cat = names_to_cat(dict(enumerate(weights.meta["categories"])), name)
        self._setup()

    def _setup(self):
        self._canvas = None
        self.fix = Fix()
        orig = self.model.transform.batch_images
        def batch_images(images, size_divisible=32):
            b = orig(images, size_divisible)
            H, W = self._canvas if self._canvas is not None else tuple(b.shape[-2:])
            assert b.shape[-2] <= H and b.shape[-1] <= W, f"canvas {(H, W)} smaller than batch {tuple(b.shape[-2:])}"
            if self.fix.conv:      # keep at least one empty stride-32 row/column past every image (see Fix)
                H, W = H + CFG["fix_margin"], W + CFG["fix_margin"]
            if (H, W) == tuple(b.shape[-2:]):
                return b
            return F.pad(b, (0, W - b.shape[-1], 0, H - b.shape[-2]))
        self.model.transform.batch_images = batch_images
        install_fix_tv(self.model, self.fix)

    @torch.inference_mode()
    def input_size(self, img):
        il, _ = self.model.transform([self.tf(img).to(DEVICE)])
        return tuple(il.image_sizes[0])

    @torch.inference_mode()
    def _run(self, imgs, canvas=None):
        self._canvas = canvas
        rpn = getattr(self.model, "rpn", None)
        if rpn is not None:
            rpn.score_thresh = CFG["fix_rpn_thresh"] if self.fix.conv else 0.0
        try:
            x = [self.tf(im).to(DEVICE) for im in imgs]
            il, _ = self.model.transform(x)
            H, W = il.tensors.shape[-2:]
            pad = [1 - (h * w) / (H * W) for h, w in il.image_sizes]
            self.fix.valid, self.fix.canvas = [tuple(s) for s in il.image_sizes], (int(H), int(W))
            out = self.model(x)
        finally:
            self._canvas = None; self.fix.valid = None
        return [_pack(o["boxes"], o["scores"], o["labels"], self.lab2cat) for o in out], pad

def _tv(builder_name, weights_name, name):
    import torchvision.models.detection as D
    return lambda: TVDet(name, getattr(D, builder_name), getattr(D, weights_name).DEFAULT)

LOADERS = {
    "detr_r50": lambda: HFDet("detr_r50", "facebook/detr-resnet-50", "DetrForObjectDetection", revision="no_timm"),
    "deformable_detr": lambda: HFDet("deformable_detr", "SenseTime/deformable-detr", "DeformableDetrForObjectDetection"),
    "frcnn_v2": _tv("fasterrcnn_resnet50_fpn_v2", "FasterRCNN_ResNet50_FPN_V2_Weights", "frcnn_v2"),
    "maskrcnn_v2": _tv("maskrcnn_resnet50_fpn_v2", "MaskRCNN_ResNet50_FPN_V2_Weights", "maskrcnn_v2"),
    "retinanet_v2": _tv("retinanet_resnet50_fpn_v2", "RetinaNet_ResNet50_FPN_V2_Weights", "retinanet_v2"),
    "fcos": _tv("fcos_resnet50_fpn", "FCOS_ResNet50_FPN_Weights", "fcos"),
    "rtdetr_r50": lambda: HFDet("rtdetr_r50", "PekingU/rtdetr_r50vd", "RTDetrForObjectDetection", fixed_size=True),
}
# Published full COCO val2017 box AP; our 500-image batch-1 AP should be within about ±2.
REF_AP = {"detr_r50": 42.0, "deformable_detr": 44.5, "frcnn_v2": 46.7, "maskrcnn_v2": 47.4,
          "retinanet_v2": 41.5, "fcos": 39.2, "rtdetr_r50": 53.1}

def load(name):
    free_mem()
    t0 = time.time()
    try:
        det = LOADERS[name]()
        det.micro = CFG["micro_batch"].get(name)
    except Exception as ex:
        print(f"✗ {name} failed to load: {type(ex).__name__}: {str(ex)[:300]}  — skipping this model")
        return None
    note = f"; runs in pieces of {det.micro} (padded to the full batch's canvas)" if det.micro else ""
    print(f"\n== {name} loaded in {time.time() - t0:.0f}s; mapped classes {len(set(det.lab2cat.values()))}/80{note}")
    return det

def free_mem():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

## 5. Metrics and batching helpers

In [ ]:
def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def greedy_match(p, q, thr, iou_min=0.5):
    ia = np.where(p["scores"] >= thr)[0]; ib = np.where(q["scores"] >= thr)[0]
    ia = ia[np.argsort(-p["scores"][ia])]
    M = iou_matrix(p["boxes"][ia], q["boxes"][ib]); used = set(); pairs = []
    for r, i in enumerate(ia):
        best, bc = -1.0, None
        for c, j in enumerate(ib):
            if c not in used and p["labels"][i] == q["labels"][j] and M[r, c] > best:
                best, bc = M[r, c], c
        if bc is not None and best >= iou_min:
            used.add(bc); pairs.append((i, ib[bc], best))
    return pairs, ia, ib

def compare(ref, new):
    """Image-level flip + detection-level change counts + max score change, ref vs new."""
    pairs, ia, ib = greedy_match(ref, new, CFG["flip_score_thr"])
    stable = sum(iou >= CFG["flip_iou"] for _, _, iou in pairs)
    moved = len(pairs) - stable
    lost, gained = len(ia) - len(pairs), len(ib) - len(pairs)
    allp, _, _ = greedy_match(ref, new, 0.05)
    dscore = max([abs(ref["scores"][i] - new["scores"][j]) for i, j, _ in allp], default=0.0)
    return {"flip": (moved + lost + gained) > 0, "stable": stable, "moved": moved, "lost": lost, "gained": gained,
            "max_score_delta": dscore}

def det_survival(ref, new, W, H):
    """For each confident ref detection: did it survive unchanged? plus its distance to the right/bottom edge."""
    pairs, ia, _ = greedy_match(ref, new, CFG["flip_score_thr"])
    ok = {i for i, _, iou in pairs if iou >= CFG["flip_iou"]}
    rows = []
    for i in ia:
        x1, y1, x2, y2 = ref["boxes"][i]
        rows.append({"changed": i not in ok, "d_right": (W - x2) / W, "d_bottom": (H - y2) / H,
                     "rel_area": (x2 - x1) * (y2 - y1) / (W * H), "score": ref["scores"][i]})
    return rows

def coco_ap(preds, idx=None):
    idx = list(range(N)) if idx is None else idx
    dets = []
    for k in idx:
        r = preds[k]; order = np.argsort(-r["scores"])[:100]
        for t in order:
            x1, y1, x2, y2 = r["boxes"][t]
            dets.append({"image_id": IDS[k], "category_id": int(r["labels"][t]),
                         "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(r["scores"][t])})
    if not dets:
        return 0.0
    with contextlib.redirect_stdout(io.StringIO()):
        dt = GT.loadRes(dets); E = COCOeval(GT, dt, "bbox"); E.params.imgIds = [IDS[k] for k in idx]
        E.evaluate(); E.accumulate(); E.summarize()
    return 100 * E.stats[0]

def run_batches(det, batches, canvas_fn=None, targets=None):
    """batches: list of index lists. canvas_fn(batch)->(H,W) or None. targets: optional list of index lists
    saying which members of each batch to record (others are companions). Returns preds, pads, seconds."""
    P, PD = [None] * N, [None] * N
    sync(); t0 = time.time()
    for bi, b in enumerate(batches):
        r, p = det.run([IMGS[i] for i in b], canvas=canvas_fn(b) if canvas_fn else None)
        keep = set(targets[bi]) if targets else set(b)
        for i, ri, pi in zip(b, r, p):
            if i in keep and P[i] is None:
                P[i], PD[i] = ri, pi
    sync()
    return P, PD, time.time() - t0

In [ ]:
B = CFG["batch_size"]
def chunks(order, k=B):
    order = list(order)
    return [order[i:i + k] for i in range(0, len(order), k)]

## 6. Part A — Fine padding sweep

In [ ]:
if RUN_PARTS["A"]:
    SW = read_csv(f"{OUT}/A_sweep.csv").to_dict("records") if os.path.exists(f"{OUT}/A_sweep.csv") else []
    done = {r["model"] for r in SW}
    NS = CFG["n_images_sweep"]
    for mname in CFG["sweep_models"]:
        if mname in done:
            print(f"== {mname}: sweep already done, skipping"); continue
        det = None; free_mem()
        det = load(mname)
        if det is None:
            continue
        full = det.fix.available
        levels = CFG["sweep_px"][det.kind]
        t0 = time.time()
        for variant, comps in [("none", ()), ("full", full)]:
            det.fix.set(comps)
            for i in range(NS):
                h, w = det.input_size(IMGS[i])
                h0, w0 = roundup(h, det.div), roundup(w, det.div)
                base = None
                for px in levels:
                    r, pad = det.run([IMGS[i]], canvas=(h0 + px, w0 + px))
                    if px == 0:
                        base = r[0]; continue
                    SW.append({"model": mname, "variant": variant, "px": px, "image_idx": i, "pad": pad[0], **compare(base, r[0])})
            print(f"  {mname} [{variant}] done ({time.time() - t0:.0f}s)")
        det.fix.set(())
        det = None; free_mem()
        pd.DataFrame(SW).to_csv(f"{OUT}/A_sweep.csv", index=False)

In [ ]:
SW = read_csv(f"{OUT}/A_sweep.csv")
sw = SW.groupby(["model", "variant", "px"]).agg(flip_rate=("flip", "mean"), med_dscore=("max_score_delta", "median"),
                                                mean_pad=("pad", "mean")).reset_index()
display(sw.pivot_table(index="px", columns=["model", "variant"], values="flip_rate").style.format("{:.1%}", na_rep="—"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for m in sw.model.unique():
    for v, ls in [("none", "o-"), ("full", "s--")]:
        d = sw[(sw.model == m) & (sw.variant == v)]
        axes[0].plot(d.px, d.flip_rate * 100, ls, label=f"{m} ({v})")
        axes[1].plot(d.px, d.med_dscore, ls, label=f"{m} ({v})")
for ax, yl in [(axes[0], "% images whose confident detections change"), (axes[1], "median max |score change|")]:
    ax.set_xscale("symlog", linthresh=1); ax.set_xlabel("padding added on right and bottom (pixels, model input)")
    ax.set_ylabel(yl); ax.legend(fontsize=8)
axes[0].set_title("Part A — the padding 'switch', without (solid) and with (dashed) the fix")
plt.tight_layout(); plt.savefig(f"{OUT}/A_sweep.png", dpi=180); plt.show()

## 7. Part B — The fix, component by component

In [ ]:
def variants_for(det):
    comps = det.fix.available
    return [("none", ())] + [(c, (c,)) for c in comps] + [("full", comps)]

if RUN_PARTS["B"]:
    FR = read_csv(f"{OUT}/B_runs.csv").to_dict("records") if os.path.exists(f"{OUT}/B_runs.csv") else []
    FI = read_csv(f"{OUT}/B_per_image.csv").to_dict("records") if os.path.exists(f"{OUT}/B_per_image.csv") else []
    done = {r["model"] for r in FR}
    bs1 = [[i] for i in range(N)]
    rand0 = chunks(np.random.default_rng(0).permutation(N))
    aspect = np.array([im.size[0] / im.size[1] for im in IMGS])
    WIDEST, TALLEST = int(np.argmax(aspect)), int(np.argmin(aspect))
    mp = chunks(range(N), CFG["batch_size"] - 2)
    maxpad = [c + [WIDEST, TALLEST] for c in mp]
    for mname in CFG["fix_models"]:
        if mname in done:
            print(f"== {mname}: already done, skipping"); continue
        det = None; free_mem()
        det = load(mname)
        if det is None:
            continue
        preds = {}
        for vname, comps in variants_for(det):
            det.fix.set(comps)
            runs = [("bs1", bs1, None), ("bs8", rand0, None)] + ([("bs8_maxpad", maxpad, mp)] if vname == "full" else [])
            for bname, batches, targets in runs:
                P, PD, sec = run_batches(det, batches, targets=targets)
                preds[(vname, bname)] = P
                ap = coco_ap(P)
                FR.append({"model": mname, "variant": vname, "batching": bname, "AP": ap,
                           "sec_per_img": sec / N, "mean_pad": float(np.mean(PD))})
                print(f"  {vname:10s} {bname:10s} AP={ap:6.2f}  pad={np.mean(PD):.3f}  {1000 * sec / N:5.1f} ms/img")
        det.fix.set(())
        ref = preds[("none", "bs1")]
        for (vname, bname), P in preds.items():
            for i in range(N):
                if bname != "bs1":      # invariance: same fix, batch 1 vs this batching
                    FI.append({"model": mname, "variant": vname, "batching": bname, "pair": "invariance",
                               **compare(preds[(vname, "bs1")][i], P[i])})
                if vname != "none" or bname != "bs1":   # fidelity: vs the library's normal batch 1
                    FI.append({"model": mname, "variant": vname, "batching": bname, "pair": "vs_default_bs1",
                               **compare(ref[i], P[i])})
        det = None; preds = None; free_mem()
        pd.DataFrame(FR).to_csv(f"{OUT}/B_runs.csv", index=False)
        pd.DataFrame(FI).to_csv(f"{OUT}/B_per_image.csv", index=False)

In [ ]:
FR = read_csv(f"{OUT}/B_runs.csv"); FI = read_csv(f"{OUT}/B_per_image.csv")
fl = FI.groupby(["model", "variant", "batching", "pair"]).agg(flip=("flip", "mean"), p95_dscore=(
    "max_score_delta", lambda v: np.percentile(v, 95))).reset_index()
ref_ap = FR[(FR.variant == "none") & (FR.batching == "bs1")].set_index("model")["AP"]
ref_t = FR[(FR.variant == "none") & (FR.batching == "bs8")].set_index("model")["sec_per_img"]
FR["dAP_vs_default_bs1"] = FR.apply(lambda r: r.AP - ref_ap[r.model], axis=1)
FR["speed_vs_default_bs8"] = FR.apply(lambda r: ref_t[r.model] / r.sec_per_img, axis=1)

inv = fl[fl.pair == "invariance"].pivot_table(index=["model", "variant"], columns="batching", values="flip")
print("INVARIANCE — % images that change between batch 1 and batch 8 under the same setting (lower = better):")
display(inv.style.format("{:.2%}", na_rep="—"))

tab = FR[FR.batching != "bs1"].merge(fl[fl.pair == "vs_default_bs1"][["model", "variant", "batching", "flip"]],
                                     on=["model", "variant", "batching"], how="left")
print("FIDELITY & SPEED at batch 8 (flip = vs the library's normal batch-1 output):")
display(tab[["model", "variant", "batching", "AP", "dAP_vs_default_bs1", "flip", "speed_vs_default_bs8"]]
        .style.format({"AP": "{:.2f}", "dAP_vs_default_bs1": "{:+.2f}", "flip": "{:.2%}", "speed_vs_default_bs8": "{:.2f}×"}))
b1 = FR[FR.batching == "bs1"].merge(fl[(fl.pair == "vs_default_bs1") & (fl.batching == "bs1")][["model", "variant", "flip"]],
                                    on=["model", "variant"], how="left")
print("Effect of each setting on the batch-1 output itself:")
display(b1[["model", "variant", "AP", "dAP_vs_default_bs1", "flip"]].style.format(
    {"AP": "{:.2f}", "dAP_vs_default_bs1": "{:+.2f}", "flip": "{:.2%}"}, na_rep="—"))

fig, ax = plt.subplots(figsize=(11, 4))
d = fl[(fl.pair == "invariance") & (fl.batching == "bs8")]
models = list(dict.fromkeys(d.model)); vs = ["none", "conv", "norm", "strides", "exactmask", "full"]
w = 0.14
for k, v in enumerate(vs):
    y = d[d.variant == v].set_index("model").reindex(models)["flip"] * 100
    ax.bar(np.arange(len(models)) + (k - 2.5) * w, y, w, label=v)
ax.set_xticks(range(len(models))); ax.set_xticklabels(models); ax.set_ylabel("% images changed, batch 1 vs batch 8")
ax.set_title("Part B — which leak matters? (each switch alone, then all)"); ax.legend(ncol=6, fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/B_attribution.png", dpi=180); plt.show()

## 8. Claims check

In [ ]:
K = CFG["claims"]; verdicts = {}
e1 = []
for m in sw.model.unique():
    d = sw[(sw.model == m) & (sw.variant == "none")].sort_values("px")
    first, last = d.flip_rate.iloc[0], d.flip_rate.iloc[-1]
    e1.append(first >= 0.5 * last)
    print(f"  E1 {m:12s}: flip at {int(d.px.iloc[0])} px = {first:.1%}, at {int(d.px.iloc[-1])} px = {last:.1%}")
verdicts["E1_step"] = f"{'✓' if all(e1) else '✗'}  {sum(e1)}/{len(e1)} models"
e2, e3, e4 = [], [], []
for m in FR.model.unique():
    f_r = fl[(fl.model == m) & (fl.variant == "full") & (fl.pair == "invariance") & (fl.batching == "bs8")].flip.iloc[0]
    f_m = fl[(fl.model == m) & (fl.variant == "full") & (fl.pair == "invariance") & (fl.batching == "bs8_maxpad")].flip.iloc[0]
    r = FR[(FR.model == m) & (FR.variant == "full") & (FR.batching == "bs8")].iloc[0]
    e2.append(max(f_r, f_m) <= K["E2_flip"]); e3.append(abs(r.dAP_vs_default_bs1) <= K["E3_dap"])
    e4.append(r.speed_vs_default_bs8 >= K["E4_speed"])
    print(f"  {m:12s} full: invariance flips {f_r:.2%} (random) / {f_m:.2%} (max-pad), "
          f"ΔAP {r.dAP_vs_default_bs1:+.2f}, speed {r.speed_vs_default_bs8:.2f}×")
verdicts["E2_invariance"] = f"{'✓' if all(e2) else '✗'}  {sum(e2)}/{len(e2)} models"
verdicts["E3_fidelity"] = f"{'✓' if all(e3) else '✗'}  {sum(e3)}/{len(e3)} models"
verdicts["E4_speed"] = f"{'✓' if all(e4) else '✗'}  {sum(e4)}/{len(e4)} models"
for k, v in verdicts.items():
    print(f"{k:16s} {v}")
with open(f"{OUT}/manifest.json", "w") as f:
    json.dump({"env": ENV, "config": CFG, "image_ids": IDS, "verdicts": verdicts}, f, indent=2, default=str)
print("saved:", sorted(os.listdir(OUT)))

## What to bring back

Upload the finished notebook. Key outputs: the sweep table and figure, the invariance table, the fidelity & speed table, the attribution figure, and the claim lines at the end.